In [47]:
# unsupervised learning using PCA (principal component analysis))

In [48]:
# importing packages
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
from sklearn.linear_model import LogisticRegression
import ipywidgets as widgets
from IPython.display import display
from sklearn.decomposition import PCA
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report

In [49]:
# data cleaning code to use at start of each model/analysis

# bring in date
train = pd.read_csv("Train1.csv", encoding='latin1')
# all numeric cols
numeric_cols = ['HOSPNUM', 'RDELAY', 'AGE', 'RSBP', 'HOURLOCAL', 'MINLOCAL', 'ONDRUG', 
    'DMAJNCHD', 'DSIDED', 'DRSISCD', 'DRSHD', 'DRSUNKD', 'DPED', 'DALIVED', 'DDEADD',
    'FLASTD', 'FDEADC', 'FU1_RECD', 'FU2_DONE', 'FU1_COMP', 'TD', 'EXPDD', 'EXPD6', 'EXPD14'
]

# all cat calls
categorical_cols = ['RCONSC', 'SEX','RSLEEP', 'RATRIAL', 'RCT', 
    'RVISINF', 'RHEP24', 'RASP3', 'RDEF1', 'RDEF2', 'RDEF3', 'RDEF4', 'RDEF5',
    'RDEF6', 'RDEF7', 'RDEF8', 'STYPE', 'DAYLOCAL', 'RXASP', 'RXHEP', 'DASP14', 
    'DASPLT', 'DLH14', 'DMH14', 'DHH14', 'DSCH', 'DIVH', 'DAP', 'DOAC', 'DGORM', 
    'DSTER', 'DCAA', 'DHAEMD', 'DCAREND', 'DTHROMB', 'DMAJNCH', 'DSIDE', 'DDIAGISC', 
    'DDIAGHA', 'DDIAGUN', 'DNOSTRK', 'DRSISC', 'DRSH', 'DRSUNK', 'DPE', 'DALIVE',
    'DPLACE', 'DDEAD', 'DDEADC', 'FDEAD', 'FRECOVER', 'FDENNIS', 'FPLACE',
    'FAP', 'FOAC', 'COUNTRY', 'CNTRYNUM', 'CMPLASP', 'CMPLHEP', 'ID', 'SET14D', 
    'ID14', 'OCCODE', 'DEAD1', 'DEAD2', 'DEAD3', 'DEAD4', 'DEAD5', 'DEAD6', 'DEAD7', 
    'DEAD8', 'H14', 'ISC14', 'NK14', 'STRK14', 'HTI14', 'PE14', 'DVT14', 'TRAN14', 'NCB14' 
]

# encoding cat variables to be numeric
train_encoded = train.copy()
encoding_maps = {}

for col in categorical_cols:
    if col in train_encoded.columns:
        train_encoded[col], mapping = pd.factorize(train_encoded[col], sort=True)
        encoding_maps[col] = dict(enumerate(mapping))
    else:
        print(f"Column '{col}' not found in DataFrame — skipping.")

train_clean1 = train_encoded


# dropping unnecessary columns
train_clean2 = train_clean1.drop(['DDEAD', 'OCCODE', 'DDEADD', 'FDEAD', 'DDEADX', 'FDEADD', 'DDEADC', 'FDEADX', 'FDEADC', 'FLASTD', 'NCCODE', 'RDATE', 'DMAJNCHX', 'DSIDEX', 'DNOSTRKX'], axis =1)

# Fill columns with median/-1 for NaN values
for col in train_clean2.columns:
    if col in numeric_cols:
        train_clean2[col].fillna(train_clean2[col].median(), inplace=True)
    else:
        train_clean2[col].fillna(-1, inplace=True)


train_clean = train_clean2.copy()

train_clean.head()

# FDEAD IS ENCODED AS N = 0, Y = 1 (there are no U's)

/var/folders/8y/9chxqdx14hb3jmz02d9_ny3r0000gn/T/ipykernel_37392/385055122.py:4: DtypeWarning: Columns (32) have mixed types. Specify dtype option on import or set low_memory=False.
  train = pd.read_csv("Train1.csv", encoding='latin1')
/var/folders/8y/9chxqdx14hb3jmz02d9_ny3r0000gn/T/ipykernel_37392/385055122.py:46: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  train_clean2[col].fillna(-1, inplace=True)
/var/folders/8y/9chxqdx14hb3jmz02d9_ny3r0000gn/T/ipykernel_37392/385055122.py:44: FutureWarning: A value is trying to be set

,ID,HOSPNUM,RDELAY,RCONSC,SEX,AGE,RSLEEP,RATRIAL,RCT,RVISINF,...,DEAD8,H14,ISC14,NK14,STRK14,HTI14,PE14,DVT14,TRAN14,NCB14
0,0,1,17,0,1,69,1,-1,1,1,...,0,0,0,0,0,0,0,0,0,0
1,1,1,10,1,1,76,1,-1,1,0,...,0,0,0,0,0,0,0,0,0,0
2,2,1,24,1,1,23,0,-1,1,0,...,0,0,0,0,0,0,0,0,0,0
3,3,1,5,0,0,83,0,-1,0,0,...,0,0,0,0,0,0,0,0,0,0
4,4,1,8,0,0,64,1,-1,1,1,...,0,0,0,0,0,0,0,0,0,0


In [50]:
# Separate features and labels
X_train = train_clean.drop(columns=['DIED'])
y_train = train_clean['DIED']

# standardize features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)


In [51]:
pca = PCA(n_components=0.95)
X_train_pca = pca.fit_transform(X_train_scaled)

In [52]:
# bring in test data
test = pd.read_csv("test1.csv", encoding='latin1')
# encoding cat variables to be numeric
test_encoded = test.copy()

encoding_maps = {}

for col in categorical_cols:
    if col in test_encoded.columns:
        test_encoded[col], mapping = pd.factorize(test_encoded[col], sort=True)
        encoding_maps[col] = dict(enumerate(mapping))

test_clean1 = test_encoded

# dropping unnecessary columns
test_clean2 = test_clean1.drop(['FLASTD', 'NCCODE', 'RDATE', 'DMAJNCHX', 'DSIDEX', 'DNOSTRKX'], axis =1)

# Fill columns with median/-1 for NaN values
for col in test_clean2.columns:
    if col in numeric_cols:
        test_clean2[col].fillna(test_clean2[col].median(), inplace=True)
    else:
        test_clean2[col].fillna(-1, inplace=True)
test_clean = test_clean2.copy()

test_clean.head()

/var/folders/8y/9chxqdx14hb3jmz02d9_ny3r0000gn/T/ipykernel_37392/3274411968.py:2: DtypeWarning: Columns (32) have mixed types. Specify dtype option on import or set low_memory=False.
  test = pd.read_csv("test1.csv", encoding='latin1')
/var/folders/8y/9chxqdx14hb3jmz02d9_ny3r0000gn/T/ipykernel_37392/3274411968.py:23: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  test_clean2[col].fillna(-1, inplace=True)
/var/folders/8y/9chxqdx14hb3jmz02d9_ny3r0000gn/T/ipykernel_37392/3274411968.py:21: FutureWarning: A value is trying to be set

,ID,HOSPNUM,RDELAY,RCONSC,SEX,AGE,RSLEEP,RATRIAL,RCT,RVISINF,...,DEAD8,H14,ISC14,NK14,STRK14,HTI14,PE14,DVT14,TRAN14,NCB14
0,0,1,43,1,0,71,0,-1,1,0,...,0,0,0,0,0,0,0,0,0,0
1,1,1,6,1,1,81,0,-1,0,0,...,0,0,0,0,0,0,0,0,0,0
2,2,4,20,1,1,78,0,-1,0,0,...,0,0,0,0,0,0,0,0,0,0
3,3,1,39,1,1,54,0,-1,1,0,...,0,0,0,0,0,0,0,0,0,0
4,4,1,4,1,0,77,0,-1,0,0,...,0,0,0,0,0,0,0,0,0,1


In [53]:
x_test_scaled = scaler.transform(test_clean)

x_test_pca = pca.transform(x_test_scaled)



In [54]:
# Separate features and labels
# x_test = test_clean
# y_test = test['DIED']

In [55]:
log_regr = LogisticRegression(solver='lbfgs')
log_regr.fit(X_train_pca, y_train)

LogisticRegression()

In [56]:
test_preds = log_regr.predict(x_test_pca)

In [57]:
from sklearn.metrics import accuracy_score
'''
if 'DIED' in test.columns:
	y_test = test['DIED']
else:
	raise KeyError("The column 'DIED' is not present in the test DataFrame.")

y_pred = test_preds
accuracy = accuracy_score(y_test, y_pred)
print(f"Test accuracy: {accuracy:.2f}")
'''

'\nif \'DIED\' in test.columns:\n\ty_test = test[\'DIED\']\nelse:\n\traise KeyError("The column \'DIED\' is not present in the test DataFrame.")\n\ny_pred = test_preds\naccuracy = accuracy_score(y_test, y_pred)\nprint(f"Test accuracy: {accuracy:.2f}")\n'

In [58]:
submission_pca = pd.DataFrame({
    'ID': test['ID'],
    'PatientDied': np.where(test_preds == 1, 'Y', 'N')
}) 
# 

In [59]:
submission_pca.to_csv('submission_pca.csv', index=False)